In [1]:

%pip install --upgrade --quiet datasets


In [2]:
from datasets import load_dataset
import pandas as pd


In [3]:

# Charger le dataset "sms_spam"
raw = load_dataset("ucirvine/sms_spam")

# Split : 4 000 pour le train, 1 000 pour le val
train_ds = raw["train"].select(range(4000))
val_ds   = raw["train"].select(range(4000, 5000))

# Afficher les features du dataset train
print(train_ds.features)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

{'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam'])}


In [4]:
from transformers import GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 n'a pas de pad_token, donc on utilise eos_token

def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],               # colonne "sms"
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
import torch
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    model_name,             # "gpt2"
    num_labels=2,           # 2 classes : spam et ham
    pad_token_id=tokenizer.eos_token_id
)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
import evaluate
import numpy as np

accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average='binary')["precision"],
        "recall":    recall.compute(predictions=preds, references=labels, average='binary')["recall"],
        "f1":        f1.compute(predictions=preds, references=labels, average='binary')["f1"]
    }


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-sms-spam",    # dossier où sauvegarder les checkpoints
    do_train=True,
    do_eval=True,
    eval_steps=500,                  # évalue toutes les 500 étapes
    save_steps=500,                  # sauvegarde toutes les 500 étapes
    logging_dir="./logs",
    logging_steps=500,               # log toutes les 500 étapes

    per_device_train_batch_size=8,   # batch size d'exemple, à ajuster selon GPU dispo
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,

    report_to=None,                  # désactive les intégrations externes
    save_total_limit=1,              # conserve uniquement le dernier checkpoint
)


weight_decay est une forme de régularisation L2 : ça “pénalise” les poids trop grands dans le modèle, pour éviter l’overfitting.

Valeur plus élevée : utile si tu as peu de données ou si tu observes du surapprentissage (train >> val).

Valeur plus faible : si ton modèle sous-apprend (train et val sont tous les deux bas) ou si tu as beaucoup de données.

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

# Lancement de l'entraînement
trainer.train()

# Évaluation finale
metrics = trainer.evaluate()
print(metrics)


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: isaacderhy18 (isaacderhy18-pst-b) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.176100
1000,0.042600


{'eval_loss': 0.054566457867622375, 'eval_accuracy': 0.99, 'eval_precision': 1.0, 'eval_recall': 0.9280575539568345, 'eval_f1': 0.9626865671641791, 'eval_runtime': 3.9247, 'eval_samples_per_second': 254.798, 'eval_steps_per_second': 31.85, 'epoch': 2.0}


Le modèle GPT-2 fine-tuné est efficace pour détecter le spam dans les SMS.
Les résultats montrent une excellente précision et un très bon rappel, adaptés à la plupart des usages pratiques.

In [14]:
# Si tu as téléchargé tout le dataset, prends les messages après les 5 000 premiers
test_ds = raw["train"].select(range(5000, 5572))  # adapte selon la taille réelle du jeu

test_tok = test_ds.map(tokenize_fn, batched=True)
test_labels = [int(x) for x in test_ds['label']] if isinstance(test_ds['label'][0], (str, int)) else test_ds['label']

# Prédiction sur le test set
test_pred = trainer.predict(test_tok)
from sklearn.metrics import classification_report

# Si test_pred.label_ids et test_pred.predictions existent :
import numpy as np
preds = np.argmax(test_pred.predictions, axis=-1)
print(classification_report(test_pred.label_ids, preds, target_names=['ham', 'spam']))


Map:   0%|          | 0/572 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

         ham       0.99      1.00      1.00       498
        spam       1.00      0.96      0.98        74

    accuracy                           0.99       572
   macro avg       1.00      0.98      0.99       572
weighted avg       0.99      0.99      0.99       572



Le modèle GPT-2 fine-tuné obtient un score d’accuracy et de F1 supérieur à 0.98 sur le test set jamais vu.
Il n’a pas de problème d’overfitting et détecte presque tous les spams sans générer de faux positifs.
Cette robustesse est confirmée par le test sur des données vraiment “neuves”.